# Prototyping Seamless Cloning (Poisson Image Editing)
This notebook tests the OpenCV seamless cloning algorithm to blend minority 
class defects (Class 1 and 2) onto defect-free steel backgrounds.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import random

### 1. Setup and Helper Functions

In [ ]:
CSV_PATH = "./dataset/train.csv"
IMG_DIR = "./dataset/train_images"

def rle_decode(rle_str, shape=(256, 1600)):
    """Decodes RLE string into a 2D binary mask."""
    s = rle_str.split()
    starts = np.asarray(s[0::2], dtype=int) - 1
    lengths = np.asarray(s[1::2], dtype=int)
    ends = starts + lengths
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        mask[lo:hi] = 1
    return mask.reshape(shape, order="F")

def extract_defect(image, mask):
    """Extracts the bounding box crop of the largest defect."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None
    
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Filter out massive or microscopic defects for testing
    if w < 10 or h < 10 or w > 800:
        return None, None
        
    defect_img = image[y:y+h, x:x+w]
    defect_mask = mask[y:y+h, x:x+w]
    
    return defect_img, defect_mask

### 2. Load Source (Defect) and Destination (Clean) Images

In [ ]:
df = pd.read_csv(CSV_PATH)

# Identify defect-free background images
all_images = set(os.listdir(IMG_DIR))
defective_images = set(df["ImageId"].unique())
clean_images = list(all_images - defective_images)

# Target Class 1 (Pitted Surface) or Class 2 (Crazing)
minority_df = df[(df["ClassId"].isin([1, 2])) & (df["EncodedPixels"].notna())]

# 1. Get Source Defect
valid_defect = False
while not valid_defect:
    source_row = minority_df.sample(1).iloc[0]
    src_img = cv2.imread(os.path.join(IMG_DIR, source_row["ImageId"]))
    src_img = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)
    
    src_mask = rle_decode(source_row["EncodedPixels"])
    defect_crop, mask_crop = extract_defect(src_img, src_mask)
    
    if defect_crop is not None:
        valid_defect = True

# 2. Get Clean Background
bg_name = random.choice(clean_images)
bg_img = cv2.imread(os.path.join(IMG_DIR, bg_name))
bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)

### 3. Apply OpenCV Seamless Cloning

In [ ]:
h, w = mask_crop.shape
# Pick a random center point that prevents the crop from going out of bounds
center_x = random.randint(w // 2 + 10, bg_img.shape[1] - w // 2 - 10)
center_y = random.randint(h // 2 + 10, bg_img.shape[0] - h // 2 - 10)
center = (center_x, center_y)

# Format the mask for OpenCV (needs to be 255 for foreground)
clone_mask = (mask_crop * 255).astype(np.uint8)

# Test both cloning modes
try:
    # NORMAL_CLONE completely replaces destination texture
    normal_clone = cv2.seamlessClone(defect_crop, bg_img, clone_mask, center, cv2.NORMAL_CLONE)
    
    # MIXED_CLONE preserves destination texture (usually better for steel grain)
    mixed_clone = cv2.seamlessClone(defect_crop, bg_img, clone_mask, center, cv2.MIXED_CLONE)
except Exception as e:
    print(f"Cloning failed (gradient calculation error): {e}")

### 4. Visualize the Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Show Isolated Defect
axes[0, 0].imshow(defect_crop)
axes[0, 0].set_title("Isolated Defect Crop (Source)")
axes[0, 0].axis("off")

# Show Clean Background
axes[0, 1].imshow(bg_img)
axes[0, 1].set_title("Clean Steel Surface (Destination)")
axes[0, 1].axis("off")

# Show Normal Clone
axes[1, 0].imshow(normal_clone)
axes[1, 0].set_title("Result: cv2.NORMAL_CLONE")
axes[1, 0].axis("off")

# Show Mixed Clone
axes[1, 1].imshow(mixed_clone)
axes[1, 1].set_title("Result: cv2.MIXED_CLONE")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()